# Principal Component Analysis: Telco Customer Churn

This notebook is the reproducible report for the five assignment questions. Run every cell from top to bottom with the provided Excel workbook. The workbook path below is the Windows path in the assignment; change `DATA_PATH` if the file is stored elsewhere. The workbook is not included in this repository, so the numerical findings and figures are generated from the user's copy at run time.

## Data and analytical choices

PCA is an unsupervised, linear method that replaces correlated measurements with orthogonal weighted combinations (principal components) ordered by the variance they explain. The analysis requires the numeric customer attributes `SeniorCitizen`, `tenure`, `MonthlyCharges`, and `TotalCharges`. `customerID` is an identifier, not a measured attribute, and `Churn` is an outcome; neither is used to fit PCA. Churn may be shown as a color in the score plot for descriptive comparison only. Standardizing attributes is necessary because their units and ranges differ; PCA on standardized variables is based on their correlation matrix.

Rows with a missing or non-numeric value in an analysis attribute are excluded from the PCA, and the count is reported below. Constant attributes are excluded because they have no variance. Review these exclusions when interpreting results.

## Assignment responses

1. **Scatter plots and correlations:** the scatterplot matrix and correlation heatmap below display pairwise relationships. The notebook identifies the strongest absolute pairwise correlation. A positive coefficient means variables tend to increase together; a negative coefficient means one tends to decrease as the other increases. Curved or clustered scatter patterns can be missed by Pearson correlation, so inspect the plots as well as the coefficients.
2. **PCA and score plots:** the notebook standardizes the selected inputs, eigendecomposes their sample covariance matrix, and plots customers' scores on the first two components. The loading table gives the coefficients defining each component.
3. **Number of components:** the Kaiser criterion (eigenvalue greater than 1 for standardized inputs) supplies the initial retained count. The scree and cumulative-variance plots are reported alongside it; consider the scree elbow and desired information retention before making a final choice. No single retention rule is definitive.
4. **Variables represented by components:** the signed loading coefficients and largest absolute loadings are listed for every component. The signs and magnitudes should be interpreted together with the correlation matrix: strongly correlated variables often contribute in the same direction, while opposed loadings can represent a contrast. Component signs are mathematically arbitrary, so reversing every sign in a component does not change its meaning.
5. **Outliers:** observations beyond the 99th-percentile chi-square cutoff for squared Mahalanobis distance in the retained score space are flagged and counted. This is a screening rule, not proof of erroneous data; inspect the observations and original measurements before drawing conclusions.

These are data-dependent questions: the cells below print the actual counts, coefficients, loadings, variance percentages, and outlier findings for the supplied workbook rather than asserting results for a different copy of the dataset.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path(r"C:\Users\Yawo Faustin AZIAKPO\Downloads\CST-570-RS-WAFn-UseC-Telco-Customer-Churn.xlsx")
FEATURES = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Workbook not found: {DATA_PATH}. Set DATA_PATH to the location of your Excel workbook."
    )

raw = pd.read_excel(DATA_PATH)
raw.columns = raw.columns.astype(str).str.strip()
missing_features = [name for name in FEATURES if name not in raw.columns]
if missing_features:
    raise ValueError(f"Workbook is missing required numeric columns: {missing_features}")

numeric = raw[FEATURES].apply(pd.to_numeric, errors="coerce")
numeric = numeric.replace([np.inf, -np.inf], np.nan)
complete = numeric.dropna()
excluded_rows = len(raw) - len(complete)
constant_features = complete.columns[complete.nunique(dropna=True) < 2].tolist()
analysis = complete.drop(columns=constant_features)
if len(analysis) < 2 or analysis.shape[1] < 2:
    raise ValueError("PCA requires at least two complete rows and two non-constant numeric features.")

print(f"Workbook rows: {len(raw):,}; complete rows used: {len(analysis):,}; rows excluded: {excluded_rows:,}")
print(f"Features used: {', '.join(analysis.columns)}")
if constant_features:
    print(f"Constant features excluded: {', '.join(constant_features)}")
print("Churn and customerID are not PCA inputs.")


## 1. Scatter plots and correlation coefficients

Each off-diagonal panel shows the customer values for a pair of features; the diagonal shows each feature's distribution. Pearson correlation summarizes linear association on a scale from -1 to +1. The heatmap annotates all coefficients so their direction and strength can be compared directly with the scatter patterns.

In [ ]:
correlations = analysis.corr()
n_features = len(analysis.columns)
fig, axes = plt.subplots(n_features, n_features, figsize=(3.2 * n_features, 3.0 * n_features))
for row, y_name in enumerate(analysis.columns):
    for col, x_name in enumerate(analysis.columns):
        ax = axes[row, col]
        if row == col:
            ax.hist(analysis[x_name], bins=25, color="steelblue", edgecolor="white")
        else:
            ax.scatter(analysis[x_name], analysis[y_name], s=8, alpha=0.25, color="steelblue")
            ax.set_title(f"r = {correlations.loc[y_name, x_name]:.2f}", fontsize=9)
        if row == n_features - 1:
            ax.set_xlabel(x_name, rotation=25, ha="right")
        if col == 0:
            ax.set_ylabel(y_name)
        ax.tick_params(labelsize=8)
fig.suptitle("Scatterplot matrix of numeric PCA inputs", y=1.01)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(n_features), labels=analysis.columns, rotation=35, ha="right")
ax.set_yticks(range(n_features), labels=analysis.columns)
for row in range(n_features):
    for col in range(n_features):
        ax.text(col, row, f"{correlations.iloc[row, col]:.2f}", ha="center", va="center", color="black")
ax.set_title("Pearson correlation matrix")
fig.colorbar(image, ax=ax, label="Correlation")
fig.tight_layout()
plt.show()

pairs = correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool)).stack()
strongest_pair = pairs.abs().idxmax()
strongest_r = pairs.loc[strongest_pair]
print("Correlation coefficients (Pearson r):")
display(correlations.round(3))
print(
    f"Strongest absolute relationship: {strongest_pair[0]} and {strongest_pair[1]} "
    f"(r = {strongest_r:.3f}, {('positive' if strongest_r > 0 else 'negative')})."
)


### Interpreting the relationship

Compare each printed coefficient with the corresponding scatter panel. A narrow upward-sloping cloud supports a strong positive linear correlation; a downward-sloping cloud supports a strong negative correlation; a diffuse cloud suggests a weaker linear association. Outlying points, subgroups, or nonlinear patterns can make a coefficient an incomplete summary. Correlation describes association, not causation.

## 2. Principal components and score plots

Each feature is centered and divided by its sample standard deviation. The eigendecomposition of the standardized sample covariance matrix is equivalent to PCA using the correlation matrix. Eigenvectors are component loadings; projecting each standardized customer row onto them produces its component scores. Eigenvalues measure the variance represented by each component.

In [ ]:
means = analysis.mean()
stds = analysis.std(ddof=1)
standardized = (analysis - means) / stds
covariance = np.cov(standardized.to_numpy(), rowvar=False, ddof=1)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
loadings = eigenvectors[:, order]
scores = standardized.to_numpy() @ loadings
explained = eigenvalues / eigenvalues.sum()
cumulative = np.cumsum(explained)
component_names = [f"PC{i + 1}" for i in range(len(eigenvalues))]

kaiser_count = max(1, int(np.sum(eigenvalues > 1)))
count_80 = int(np.searchsorted(cumulative, 0.80) + 1)
count_90 = int(np.searchsorted(cumulative, 0.90) + 1)
variance_table = pd.DataFrame(
    {"Eigenvalue": eigenvalues, "Variance explained (%)": explained * 100,
     "Cumulative variance (%)": cumulative * 100}, index=component_names
)
print(f"Kaiser criterion (eigenvalue > 1): retain {kaiser_count} component(s).")
print(f"Components needed for at least 80% / 90% variance: {count_80} / {count_90}.")
display(variance_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(range(1, len(eigenvalues) + 1), eigenvalues, marker="o")
axes[0].axhline(1, color="gray", linestyle="--", label="Kaiser cutoff (1)")
axes[0].set(title="Scree plot", xlabel="Principal component", ylabel="Eigenvalue")
axes[0].set_xticks(range(1, len(eigenvalues) + 1))
axes[0].legend()
axes[1].plot(range(1, len(cumulative) + 1), cumulative * 100, marker="o")
axes[1].axhline(80, color="gray", linestyle="--", label="80% reference")
axes[1].set(title="Cumulative variance explained", xlabel="Number of components", ylabel="Variance (%)", ylim=(0, 105))
axes[1].set_xticks(range(1, len(cumulative) + 1))
axes[1].legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
if "Churn" in raw.columns:
    churn = raw.loc[analysis.index, "Churn"].astype(str)
    for label in sorted(churn.unique()):
        mask = churn.to_numpy() == label
        ax.scatter(scores[mask, 0], scores[mask, 1], s=12, alpha=0.4, label=f"Churn: {label}")
    ax.legend()
else:
    ax.scatter(scores[:, 0], scores[:, 1], s=12, alpha=0.4, color="steelblue")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
ax.set_title("Customer PCA score plot (Churn is descriptive only)")
ax.axhline(0, color="gray", linewidth=0.7)
ax.axvline(0, color="gray", linewidth=0.7)
fig.tight_layout()
plt.show()

print(
    f"PC1 and PC2 together explain {cumulative[min(1, len(cumulative) - 1)] * 100:.1f}% "
    "of the variance. Use the scree elbow and variance target alongside the Kaiser result; "
    "retaining more components preserves more information but reduces the dimensionality less."
)


## 3–4. Component count and variables represented

The Kaiser count is a reproducible starting recommendation, not an automatic universal answer. Retain that many components when the scree elbow and the assignment's acceptable information loss support it; if a larger cumulative-variance target is important, report that alternative. The eigenvalue table and plots expose this trade-off.

Loadings are the weights in the linear combination of standardized inputs. Large absolute weights identify variables most represented by a component. A component with similar-signed weights for a strongly correlated pair captures their shared variation; opposite signs capture a contrast. The correlation heatmap provides the pairwise context, while PCA components can combine several variables at once.

In [ ]:
loading_table = pd.DataFrame(loadings, index=analysis.columns, columns=component_names)
print("Signed loading coefficients:")
display(loading_table.round(3))
for component in component_names:
    largest = loading_table[component].abs().sort_values(ascending=False)
    selected = largest[largest >= 0.30]
    if selected.empty:
        selected = largest.head(2)
    description = "; ".join(
        f"{name} ({loading_table.loc[name, component]:+.3f})" for name in selected.index
    )
    print(f"{component} explains {explained[component_names.index(component)] * 100:.1f}% of variance; largest loadings: {description}.")

fig, ax = plt.subplots(figsize=(max(7, len(component_names) * 1.5), 4.5))
image = ax.imshow(loading_table, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(component_names)), labels=component_names)
ax.set_yticks(range(len(analysis.columns)), labels=analysis.columns)
for row in range(len(analysis.columns)):
    for col in range(len(component_names)):
        ax.text(col, row, f"{loading_table.iloc[row, col]:.2f}", ha="center", va="center")
ax.set_title("PCA loadings")
fig.colorbar(image, ax=ax, label="Loading")
fig.tight_layout()
plt.show()


## 5. Outlier screening

The screening below uses squared Mahalanobis distance across the Kaiser-retained component scores. Because PCA scores are uncorrelated, the squared distance is the sum of each retained score squared divided by its eigenvalue. A chi-square 99th-percentile cutoff is used, with degrees of freedom equal to the retained component count. These four-feature data require at most four degrees of freedom. This flags unusual multivariate profiles, not necessarily data errors; unusually long tenure or high charges can be valid customers.

In [ ]:
chi_square_99 = {1: 6.635, 2: 9.210, 3: 11.345, 4: 13.277}
retained = np.arange(kaiser_count)
distance_squared = np.sum(scores[:, retained] ** 2 / eigenvalues[retained], axis=1)
cutoff = chi_square_99[kaiser_count]
outlier_mask = distance_squared > cutoff
print(
    f"Flagged {int(outlier_mask.sum())} of {len(analysis)} complete customers "
    f"({outlier_mask.mean() * 100:.2f}%) above the 99% chi-square cutoff "
    f"of {cutoff:.3f} for {kaiser_count} retained component(s)."
)
if outlier_mask.any():
    source_rows = analysis.index.to_numpy()[outlier_mask] + 2
    outlier_table = pd.DataFrame(
        {"Workbook row (header is row 1)": source_rows,
         "Squared Mahalanobis distance": distance_squared[outlier_mask]}
    ).sort_values("Squared Mahalanobis distance", ascending=False)
    print("Ten largest flagged observations (workbook row numbers):")
    display(outlier_table.head(10).round(3))
else:
    print("No complete observations exceeded this screening threshold.")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scores[~outlier_mask, 0], scores[~outlier_mask, 1], s=12, alpha=0.35, label="Within cutoff")
if outlier_mask.any():
    ax.scatter(scores[outlier_mask, 0], scores[outlier_mask, 1], s=35, marker="x", color="crimson", label="Flagged")
ax.set_xlabel(f"PC1 ({explained[0] * 100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1] * 100:.1f}% variance)")
ax.set_title("PCA scores with multivariate outlier flags")
ax.legend()
fig.tight_layout()
plt.show()


## Conclusion and limitations

Use the printed correlation coefficients and scatterplots to describe pairwise patterns, then use the eigenvalue and cumulative-variance output to justify a retained component count. The loading table names the numeric attributes defining each retained component, and the outlier output states the count and workbook rows flagged for review. Insert or discuss these generated results in the written submission after running the notebook on the assignment workbook.

PCA maximizes variance, not prediction of churn. It assumes that linear combinations and Euclidean variation in the standardized numeric attributes are useful summaries. Treating the binary `SeniorCitizen` field as numeric also imposes a numeric distance interpretation. Categorical service/contract/payment fields are deliberately not one-hot encoded here; adding them would change the feature space, correlation structure, and component interpretation. The outlier threshold is approximate under multivariate-normal assumptions, so it is a diagnostic rather than a definitive judgment.

## References

Hotelling, H. (1933). Analysis of a complex of statistical variables into principal components. *Journal of Educational Psychology, 24*(6), 417–441; *24*(7), 498–520. https://doi.org/10.1037/h0071325

Jolliffe, I. T., & Cadima, J. (2016). Principal component analysis: A review and recent developments. *Philosophical Transactions of the Royal Society A: Mathematical, Physical and Engineering Sciences, 374*(2065), 20150202. https://doi.org/10.1098/rsta.2015.0202

Kaiser, H. F. (1960). The application of electronic computers to factor analysis. *Educational and Psychological Measurement, 20*(1), 141–151. https://doi.org/10.1177/001316446002000116